<a href="https://colab.research.google.com/github/arcctg/kpi-ml-lab1/blob/main/task_1_ukrstat_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Додайємо всі потрібні (і можливо потрібні) для обробки даних імпорти

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statistics
import re

Завантажуєм датасет, скачаний з ukrstat та закинутий в гітхаб репо для зручності. Датасет показує динаміку цін

In [3]:
url = 'https://raw.githubusercontent.com/arcctg/kpi-ml-lab1/main/ukrstat_prices.csv'

try:
    df = pd.read_csv(url)
    print("Data was loaded successfully:")
    display(df.head())
except Exception as e:
    print(f"Error on load: {e}")

Data was loaded successfully:


,Показник,Базисний період,Територіальний розріз,Тип товарів і послуг,Періодичність,Одиниця виміру,Масштабування,Кількість десяткових знаків,Спеціальне значення даних,Примітки часового ряду,...,2025-M04,2025-M05,2025-M06,2025-M07,2025-M08,2025-M09,2025-M10,2025-M11,2025-M12,2026-M01
0,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Птиця (тушки курячі),Місячна,Кілограм,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,98.15,107.22,113.94,115.55,117.95,118.03,119.98,117.93,116.20,112.47
1,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Сигарети без фільтру,Місячна,Пачка (20 шт),Одиниці,Два,Явищ не було,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,"Молоко пастеризоване жирністю до 2,6% включно",Місячна,1000 г,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,47.71,47.39,47.18,47.10,47.72,47.45,49.20,49.05,49.56,49.95
3,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Судинорозширювальні засоби імпортні,Місячна,"10 табл., капсул",Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,82.99,83.16,83.94,85.89,88.92,95.83,96.36,92.66,89.29,88.39
4,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Сигарети з фільтром медіум класу,Місячна,Пачка (20 шт),Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,111.33,113.69,116.52,119.36,121.91,123.37,126.06,128.15,130.70,133.03


Розмір датасета велий як видно, тому треба його зменшити

In [20]:
df.shape

(2449, 467)

Для почтку відільтруємо за територією та показником, як видно нижче територія включає як всю Україну так і області. Показниках половину займають індекси

In [11]:
df.head(10)

,Показник,Базисний період,Територіальний розріз,Тип товарів і послуг,Періодичність,Одиниця виміру,Масштабування,Кількість десяткових знаків,Спеціальне значення даних,Примітки часового ряду,...,2025-M04,2025-M05,2025-M06,2025-M07,2025-M08,2025-M09,2025-M10,2025-M11,2025-M12,2026-M01
0,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Птиця (тушки курячі),Місячна,Кілограм,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,98.15,107.22,113.94,115.55,117.95,118.03,119.98,117.93,116.20,112.47
1,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Сигарети без фільтру,Місячна,Пачка (20 шт),Одиниці,Два,Явищ не було,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,"Молоко пастеризоване жирністю до 2,6% включно",Місячна,1000 г,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,47.71,47.39,47.18,47.10,47.72,47.45,49.20,49.05,49.56,49.95
3,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Судинорозширювальні засоби імпортні,Місячна,"10 табл., капсул",Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,82.99,83.16,83.94,85.89,88.92,95.83,96.36,92.66,89.29,88.39
4,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Сигарети з фільтром медіум класу,Місячна,Пачка (20 шт),Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,111.33,113.69,116.52,119.36,121.91,123.37,126.06,128.15,130.70,133.03
5,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Картопля,Місячна,Кілограм,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,30.63,31.95,34.73,24.89,21.24,17.93,16.85,15.79,14.76,15.51
6,Середні споживчі ціни на товари (послуги),Не застосовується,Вінницька,Риба морожена,Місячна,Кілограм,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,190.92,193.71,201.76,202.53,203.39,208.96,211.31,214.37,214.70,224.69
7,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Хліб пшеничний з борошна першого ґатунку,Місячна,Кілограм,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,45.06,45.38,46.19,46.31,46.39,47.07,47.68,48.26,48.81,49.88
8,Середні споживчі ціни на товари (послуги),Не застосовується,Вінницька,Крупи манні,Місячна,Кілограм,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,25.38,27.00,28.96,28.96,29.16,29.41,29.12,29.06,28.56,29.09
9,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Газ скраплений для автомобілів,Місячна,Літр,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,35.65,35.03,34.33,34.39,34.31,34.26,34.25,34.48,36.42,38.05


In [12]:
df.tail(10)

,Показник,Базисний період,Територіальний розріз,Тип товарів і послуг,Періодичність,Одиниця виміру,Масштабування,Кількість десяткових знаків,Спеціальне значення даних,Примітки часового ряду,...,2025-M04,2025-M05,2025-M06,2025-M07,2025-M08,2025-M09,2025-M10,2025-M11,2025-M12,2026-M01
2439,Індекс споживчих цін,До попереднього місяця,Севастополь,Ресторани та готелі,Місячна,Відсоток,Одиниці,Один,NaN,1. З 2001 року. 2. За 2014-2021 роки дані наве...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2440,Індекс споживчих цін,До попереднього місяця,Севастополь,Різні товари та послуги,Місячна,Відсоток,Одиниці,Один,NaN,1. З 2001 року. 2. За 2014-2021 роки дані наве...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2441,Індекс споживчих цін,Рік до попереднього року,Україна,Індекс споживчих цін,Річна,Відсоток,Одиниці,Один,NaN,1. З 1993 року. 2. За 2014 рік дані наведено б...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2442,Індекс споживчих цін,Рік до попереднього року,Україна,Охорона здоров’я,Річна,Відсоток,Одиниці,Один,NaN,1. З 2002 року. 2. За 2014 рік дані наведено б...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2443,Індекс споживчих цін,Рік до попереднього року,Україна,"Предмети домашнього вжитку, побутова техніка т...",Річна,Відсоток,Одиниці,Один,NaN,1. З 2002 року. 2. За 2014 рік дані наведено б...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2444,Індекс споживчих цін,Рік до попереднього року,Україна,"Житло, вода, електроенергія, газ та інші види ...",Річна,Відсоток,Одиниці,Один,NaN,1. З 2002 року. 2. За 2014 рік дані наведено б...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2445,Індекс споживчих цін,Рік до попереднього року,Україна,Одяг і взуття,Річна,Відсоток,Одиниці,Один,NaN,1. З 2002 року. 2. За 2014 рік дані наведено б...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2446,Індекс споживчих цін,Рік до попереднього року,Україна,Відпочинок і культура,Річна,Відсоток,Одиниці,Один,NaN,1. З 2002 року. 2. За 2014 рік дані наведено б...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2447,Індекс споживчих цін,Рік до попереднього року,Україна,Зв’язок,Річна,Відсоток,Одиниці,Один,NaN,1. З 2002 року. 2. За 2014 рік дані наведено б...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2448,Індекс споживчих цін,Рік до попереднього року,Україна,Транспорт,Річна,Відсоток,Одиниці,Один,NaN,1. З 2002 року. 2. За 2014 рік дані наведено б...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Залишаємо лише Україну та Середні споживчі ціни на товари (послуги)

In [21]:
df[
    (df['Територіальний розріз'] == 'Україна') &
    (df['Показник'] == 'Середні споживчі ціни на товари (послуги)')
]

,Показник,Базисний період,Територіальний розріз,Тип товарів і послуг,Періодичність,Одиниця виміру,Масштабування,Кількість десяткових знаків,Спеціальне значення даних,Примітки часового ряду,...,2025-M04,2025-M05,2025-M06,2025-M07,2025-M08,2025-M09,2025-M10,2025-M11,2025-M12,2026-M01
0,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Птиця (тушки курячі),Місячна,Кілограм,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,98.15,107.22,113.94,115.55,117.95,118.03,119.98,117.93,116.20,112.47
1,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Сигарети без фільтру,Місячна,Пачка (20 шт),Одиниці,Два,Явищ не було,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,"Молоко пастеризоване жирністю до 2,6% включно",Місячна,1000 г,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,47.71,47.39,47.18,47.10,47.72,47.45,49.20,49.05,49.56,49.95
3,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Судинорозширювальні засоби імпортні,Місячна,"10 табл., капсул",Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,82.99,83.16,83.94,85.89,88.92,95.83,96.36,92.66,89.29,88.39
4,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Сигарети з фільтром медіум класу,Місячна,Пачка (20 шт),Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,111.33,113.69,116.52,119.36,121.91,123.37,126.06,128.15,130.70,133.03
5,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Картопля,Місячна,Кілограм,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,30.63,31.95,34.73,24.89,21.24,17.93,16.85,15.79,14.76,15.51
7,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Хліб пшеничний з борошна першого ґатунку,Місячна,Кілограм,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,45.06,45.38,46.19,46.31,46.39,47.07,47.68,48.26,48.81,49.88
9,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Газ скраплений для автомобілів,Місячна,Літр,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,35.65,35.03,34.33,34.39,34.31,34.26,34.25,34.48,36.42,38.05
12,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Стоматологічні послуги,Місячна,Одиниця,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,1264.74,1280.71,1297.96,1314.15,1323.65,1332.81,1351.46,1362.90,1379.62,1415.78
13,Середні споживчі ціни на товари (послуги),Не застосовується,Україна,Бензин А-92,Місячна,Літр,Одиниці,Два,NaN,1. З 2017 року. 2. Дані за одиницю вимірювання...,...,52.29,51.63,53.48,56.35,56.16,55.70,55.90,55.70,55.91,56.69
